# Información cuántica: qubits, puertas y entropía de Von Neumann

Exploración computacional de los conceptos básicos de información cuántica:
representación de qubits, puertas cuánticas, entrelazamiento y entropía de Von Neumann.

**Artículo relacionado:** `05/04-informacion-cuantica.md`

In [ ]:
import math
import numpy as np
from fractions import Fraction

## 1. Representación de qubits y estados cuánticos

In [ ]:
# Un qubit es un vector complejo en C^2 de norma 1: |ψ⟩ = α|0⟩ + β|1⟩
# con |α|² + |β|² = 1

def qubit(alpha, beta):
    """Crea un qubit y verifica que es unitario."""
    v = np.array([alpha, beta], dtype=complex)
    norm = np.linalg.norm(v)
    if not np.isclose(norm, 1.0):
        raise ValueError(f"El vector no es unitario: norma = {norm:.4f}")
    return v


def measure_prob(state):
    """Probabilidades de medir 0 y 1 en un qubit."""
    return abs(state[0])**2, abs(state[1])**2


def dirac_notation(state):
    """Representa el estado en notación de Dirac."""
    a, b = state[0], state[1]
    terms = []
    if not np.isclose(a, 0):
        terms.append(f"({a:.3f})|0⟩")
    if not np.isclose(b, 0):
        terms.append(f"({b:.3f})|1⟩")
    return " + ".join(terms) if terms else "0"


# Estados fundamentales
ket0 = qubit(1, 0)  # |0⟩
ket1 = qubit(0, 1)  # |1⟩
plus  = qubit(1/math.sqrt(2),  1/math.sqrt(2))  # |+⟩
minus = qubit(1/math.sqrt(2), -1/math.sqrt(2))  # |−⟩
ket_i = qubit(1/math.sqrt(2),  1j/math.sqrt(2)) # |i⟩

estados = [("  |0⟩", ket0), ("  |1⟩", ket1), ("  |+⟩", plus), ("  |−⟩", minus), (" |i⟩", ket_i)]

print("Estados cuánticos fundamentales:")
print(f"{'Estado':<8} {'α':<20} {'β':<20} {'P(0)':<8} {'P(1)'}")
print('-' * 65)
for name, state in estados:
    p0, p1 = measure_prob(state)
    print(f"{name:<8} {str(state[0]):<20} {str(state[1]):<20} {p0:<8.4f} {p1:.4f}")

print()
print("Verificación ortogonalidad:")
print(f"  ⟨0|1⟩ = {np.dot(ket0.conj(), ket1):.4f}  (debe ser 0)")
print(f"  ⟨+|−⟩ = {np.dot(plus.conj(), minus):.4f}  (debe ser 0)")
print(f"  ⟨+|0⟩ = {np.dot(plus.conj(), ket0):.4f}  (no ortogonales)")

### Ejercicio 1.1: Verificar superposición
Comprueba que el estado $|\psi\rangle = \frac{\sqrt{3}}{2}|0\rangle + \frac{1}{2}|1\rangle$
es válido y calcula sus probabilidades de medición.

In [ ]:
# Tu solución:
# psi = qubit(math.sqrt(3)/2, 1/2)
# p0, p1 = measure_prob(psi)
# print(f"|ψ⟩ = {dirac_notation(psi)}")
# print(f"P(0) = {p0:.4f}, P(1) = {p1:.4f}")
# Suma: {p0 + p1:.4f} (debe ser 1)

print("TODO: crea el qubit y verifica las probabilidades")

# Solución de referencia (comentada):
# psi = qubit(math.sqrt(3)/2, 0.5)
# |ψ⟩ = (0.866)|0⟩ + (0.500)|1⟩
# P(0) = 0.7500, P(1) = 0.2500 → suman 1 ✓

## 2. Puertas cuánticas de un qubit

In [ ]:
# Las puertas cuánticas son matrices unitarias U (U†U = I)

# Puerta Hadamard: H = 1/√2 [[1,1],[1,-1]]
H = np.array([[1, 1], [1, -1]], dtype=complex) / math.sqrt(2)

# Puerta Pauli-X (NOT cuántico): X = [[0,1],[1,0]]
X = np.array([[0, 1], [1, 0]], dtype=complex)

# Puerta Pauli-Y: Y = [[0,-i],[i,0]]
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)

# Puerta Pauli-Z: Z = [[1,0],[0,-1]]
Z = np.array([[1, 0], [0, -1]], dtype=complex)

# Puerta de fase S: S = [[1,0],[0,i]]
S = np.array([[1, 0], [0, 1j]], dtype=complex)


def apply_gate(gate, state):
    """Aplica una puerta cuántica a un estado."""
    new_state = gate @ state
    assert np.isclose(np.linalg.norm(new_state), 1.0), "Puerta no preserva norma"
    return new_state


def is_unitary(gate, tol=1e-10):
    """Verifica que la puerta sea unitaria: U†U = I."""
    product = gate.conj().T @ gate
    return np.allclose(product, np.eye(len(gate)), atol=tol)


print("Verificación de unitariedad:")
for name, gate in [("H", H), ("X", X), ("Y", Y), ("Z", Z), ("S", S)]:
    print(f"  {name} unitaria: {is_unitary(gate)}")

print()
print("Aplicando Hadamard:")
print(f"  H|0⟩ = {dirac_notation(apply_gate(H, ket0))}   (= |+⟩)")
print(f"  H|1⟩ = {dirac_notation(apply_gate(H, ket1))}   (= |−⟩)")
print(f"  H|+⟩ = {dirac_notation(apply_gate(H, plus))}   (= |0⟩)")
print(f"  H|−⟩ = {dirac_notation(apply_gate(H, minus))}  (= |1⟩)")
print()

# H² = I (propiedad fundamental)
H_squared = H @ H
print(f"H² = I: {np.allclose(H_squared, np.eye(2))}")
print()
print("Aplicando X (NOT cuántico):")
print(f"  X|0⟩ = {dirac_notation(apply_gate(X, ket0))}  (= |1⟩)")
print(f"  X|1⟩ = {dirac_notation(apply_gate(X, ket1))}  (= |0⟩)")
print(f"  X|+⟩ = {dirac_notation(apply_gate(X, plus))}  (= |+⟩, |+⟩ es autoestado de X)")

### Ejercicio 2.1: Circuito Hadamard-Z-Hadamard
Verifica que $HZH = X$ componiendo las puertas matricialmente.

In [ ]:
# Tu solución:
# HZH = H @ Z @ H
# ¿Es igual a X?

# Pista: en quantum computing, el orden de las operaciones es de derecha a izquierda
# (primero se aplica H, luego Z, luego H)

# Tu código aquí:
# resultado = H @ Z @ H
# print(f"HZH =\n{np.round(resultado, 3)}")
# print(f"X =\n{X}")
# print(f"HZH = X: {np.allclose(resultado, X)}")

print("TODO: verifica que HZH = X")

# Solución de referencia:
# HZH = [[0,1],[1,0]] = X ✓
# Esto muestra que las puertas Pauli no son independientes:
# conjugar Z con Hadamard lo convierte en X (cambio de base computacional ↔ Hadamard)

## 3. Entrelazamiento y estados de Bell

In [ ]:
# Sistema de 2 qubits: espacio de Hilbert C^4
# Base computacional: |00⟩, |01⟩, |10⟩, |11⟩

def tensor_product(state_a, state_b):
    """Producto tensorial de dos estados cuánticos."""
    return np.kron(state_a, state_b)


def is_entangled(state_2q, tol=1e-10):
    """
    Detecta si un estado de 2 qubits es entrelazado.
    Un estado es separable si perm = α⊗β, lo que implica
    state[0]*state[3] = state[1]*state[2] (condición de rango 1).
    """
    # Reorganizar como matriz 2x2
    M = state_2q.reshape(2, 2)
    # El estado es separable sii rank(M) = 1
    rank = np.linalg.matrix_rank(M, tol=tol)
    return rank > 1


# Puerta CNOT (Controlled-NOT) en base |00⟩,|01⟩,|10⟩,|11⟩
CNOT = np.array([
    [1, 0, 0, 0],
    [0, 1, 0, 0],
    [0, 0, 0, 1],
    [0, 0, 1, 0]
], dtype=complex)

# H en el primer qubit: H⊗I
H_I = np.kron(H, np.eye(2))

# Estado inicial |00⟩
ket00 = tensor_product(ket0, ket0)

# Circuito de Bell: aplicar H⊗I al primer qubit, luego CNOT
# Paso 1: H|0⟩ ⊗ |0⟩ = |+⟩ ⊗ |0⟩ = (|00⟩ + |10⟩)/√2
step1 = H_I @ ket00
# Paso 2: CNOT entrelaza los qubits
bell_phi_plus = CNOT @ step1

# Los 4 estados de Bell
bell_states = {
    '|Φ+⟩': CNOT @ (H_I @ tensor_product(ket0, ket0)),
    '|Φ-⟩': CNOT @ (H_I @ tensor_product(ket0, ket1)),
    '|Ψ+⟩': CNOT @ (H_I @ tensor_product(ket1, ket0)),
    '|Ψ-⟩': CNOT @ (H_I @ tensor_product(ket1, ket1)),
}

print("Estados de Bell (maximally entangled):")
base_labels = ['|00⟩', '|01⟩', '|10⟩', '|11⟩']
for name, state in bell_states.items():
    terms = [f"{state[i]:+.3f}{base_labels[i]}" for i in range(4) if not np.isclose(state[i], 0)]
    print(f"  {name} = {'  '.join(terms)}")
    print(f"        Entrelazado: {is_entangled(state)}")

print()
# Estado producto: no entrelazado
estado_producto = tensor_product(plus, ket0)
print(f"Estado producto |+⟩⊗|0⟩: entrelazado = {is_entangled(estado_producto)} (debe ser False)")
print()
print("Colapso de medida en |Φ+⟩ = (|00⟩+|11⟩)/√2:")
print("  Si medimos el qubit 1 y obtenemos |0⟩ → estado colapsa a |00⟩")
print("  Si medimos el qubit 1 y obtenemos |1⟩ → estado colapsa a |11⟩")
print("  Los resultados están perfectamente correlacionados, incluso a distancia.")

### Ejercicio 3.1: Distinguir entrelazado de no-entrelazado
¿Cuál de los siguientes estados está entrelazado?

In [ ]:
# Analiza cada estado: ¿es entrelazado?
candidates = [
    ("(|00⟩+|11⟩)/√2",     np.array([1, 0, 0, 1], dtype=complex) / math.sqrt(2)),
    ("(|00⟩+|01⟩)/√2",     np.array([1, 1, 0, 0], dtype=complex) / math.sqrt(2)),
    ("(|00⟩+|01⟩+|10⟩+|11⟩)/2", np.array([1, 1, 1, 1], dtype=complex) / 2),
    ("(|00⟩-|11⟩)/√2",     np.array([1, 0, 0, -1], dtype=complex) / math.sqrt(2)),
]

# Tu solución:
# for name, state in candidates:
#     print(f"{name}: entrelazado = {is_entangled(state)}")
# Explica por qué cada resultado tiene sentido.

print("TODO: comprueba qué estados están entrelazados y explica el patrón")

# Solución de referencia:
# (|00⟩+|11⟩)/√2 → entrelazado (estado de Bell |Φ+⟩)
# (|00⟩+|01⟩)/√2 → NO entrelazado: = |0⟩ ⊗ (|0⟩+|1⟩)/√2 = |0⟩|+⟩
# (|00⟩+|01⟩+|10⟩+|11⟩)/2 → NO entrelazado: = |+⟩ ⊗ |+⟩
# (|00⟩-|11⟩)/√2 → entrelazado (estado de Bell |Φ-⟩)

## 4. Matrices densidad y entropía de Von Neumann

In [ ]:
def density_matrix(state):
    """Matriz densidad de un estado puro: ρ = |ψ⟩⟨ψ|."""
    return np.outer(state, state.conj())


def von_neumann_entropy(rho, base=2, tol=1e-12):
    """
    Entropía de Von Neumann: S(ρ) = -tr(ρ log ρ) = -Σ λ_i log λ_i.
    Para base=2, la entropía se mide en qubits (ebits).
    """
    eigenvalues = np.linalg.eigvalsh(rho)
    eigenvalues = eigenvalues[eigenvalues > tol]  # eliminar valores negativos por error numérico
    if base == 2:
        return -np.sum(eigenvalues * np.log2(eigenvalues))
    else:
        return -np.sum(eigenvalues * np.log(eigenvalues))


def partial_trace_b(rho_ab):
    """
    Traza parcial sobre el subsistema B: ρ_A = tr_B(ρ_AB).
    Asume sistema de 2 qubits (dimensión 4).
    """
    rho_ab = rho_ab.reshape(2, 2, 2, 2)  # (a, b, a', b')
    # ρ_A[a,a'] = Σ_b ρ_AB[a,b,a',b]
    return np.einsum('abab->...', rho_ab.reshape(4, 4).reshape(2, 2, 2, 2).transpose(0, 2, 1, 3)).reshape(2,2) 


def partial_trace(rho_ab, trace_out='B'):
    """Traza parcial para sistema de 2 qubits (dim 4×4)."""
    n = 2  # dimensión de cada qubit
    rho = rho_ab.reshape(n, n, n, n)  # ρ[i_A, i_B, j_A, j_B]
    if trace_out == 'B':
        # ρ_A[i_A, j_A] = Σ_{i_B} ρ[i_A, i_B, j_A, i_B]
        return np.einsum('ijkj->ik', rho)
    else:
        # ρ_B[i_B, j_B] = Σ_{i_A} ρ[i_A, i_B, i_A, j_B]
        return np.einsum('iijk->jk', rho)


print("Entropía de Von Neumann para distintos estados:")
print()

# Estado puro |0⟩: S=0
rho_pure = density_matrix(ket0)
print(f"Estado puro |0⟩: S(ρ) = {von_neumann_entropy(rho_pure):.4f} bits (debe ser 0)")

# Mezcla clásica 50/50: S=1
rho_mixed = 0.5 * density_matrix(ket0) + 0.5 * density_matrix(ket1)
print(f"Mezcla ½|0⟩⟨0|+½|1⟩⟨1|: S(ρ) = {von_neumann_entropy(rho_mixed):.4f} bits (debe ser 1)")

# Mezcla general p|0⟩⟨0| + (1-p)|1⟩⟨1|
print()
print("Entropía de Von Neumann de ρ = p|0⟩⟨0| + (1-p)|1⟩⟨1|:")
print(f"{'p':<8} {'S(ρ) bits':<15} {'H_b(p) Shannon':<15} {'Coinciden'}")
print('-' * 50)
for p in [0.0, 0.1, 0.25, 0.5, 0.75, 0.9, 1.0]:
    rho_p = p * density_matrix(ket0) + (1-p) * density_matrix(ket1)
    s = von_neumann_entropy(rho_p)
    hb = -(p*math.log2(p) + (1-p)*math.log2(1-p)) if 0 < p < 1 else 0
    print(f"{p:<8.2f} {s:<15.4f} {hb:<15.4f} {np.isclose(s, hb)}") 

print()
print("Para mezclas en la base computacional, S(ρ) = H_b(p) (entropía de Shannon)")
print("Pero la entropía de Von Neumann es más general: captura entrelazamiento.")

### Ejercicio 4.1: Entropía de entrelazamiento del estado de Bell

Para el estado de Bell $|\Phi^+\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$:
- Calcula la matriz densidad reducida $\rho_A = \text{tr}_B(|\Phi^+\rangle\langle\Phi^+|)$
- Calcula $S(\rho_A)$ — ¿qué indica este valor sobre el entrelazamiento?

In [ ]:
# Estado de Bell |Φ+⟩
phi_plus = bell_states['|Φ+⟩']

# Tu solución:
# rho_bell = np.outer(phi_plus, phi_plus.conj())  # ρ_AB = |Φ+⟩⟨Φ+|
# rho_A = partial_trace(rho_bell, trace_out='B')   # traza parcial sobre B
# print("ρ_A =", np.round(rho_A, 3))
# print(f"S(ρ_A) = {von_neumann_entropy(rho_A):.4f} bits")
# ¿Es ρ_A pura o mixta? ¿Qué implica S=1 bit?

print("TODO: calcula la entropía de entrelazamiento del estado de Bell")

# Solución de referencia:
# ρ_AB = |Φ+⟩⟨Φ+| (4×4)
# ρ_A = tr_B(ρ_AB) = ½|0⟩⟨0| + ½|1⟩⟨1| = I/2 (mezcla máxima)
# S(ρ_A) = 1 bit = entropía máxima para 1 qubit
# Interpretación: si solo se accede al qubit A ignorando B,
# el estado parece completamente aleatorio (S=1) aunque el estado conjunto sea puro (S=0).
# Esta discrepancia es la "entropía de entrelazamiento" = 1 ebit.

## 5. Teleportación cuántica (protocolo)

In [ ]:
def quantum_teleportation(psi):
    """
    Simula el protocolo de teleportación cuántica.
    
    Alice quiere enviar |ψ⟩ a Bob.
    Recursos: 1 par de Bell compartido + 2 bits clásicos.
    
    Esquema:
    - Estado inicial: |ψ⟩_A ⊗ |Φ+⟩_AB (3 qubits)
    - Alice aplica CNOT (qubit ψ como control, qubit A como objetivo)
    - Alice aplica H a su qubit ψ
    - Alice mide sus 2 qubits → obtiene bits (m1, m2) ∈ {00,01,10,11}
    - Alice envía (m1, m2) a Bob (canal clásico)
    - Bob aplica X^m2 Z^m1 a su qubit B → obtiene |ψ⟩
    """
    # Estado inicial: |ψ⟩ ⊗ |Φ+⟩ (3 qubits, índices: ψ, A, B)
    phi_plus = np.array([1, 0, 0, 1], dtype=complex) / math.sqrt(2)
    initial = np.kron(psi, phi_plus)  # dim 8
    
    # Puertas de 3 qubits
    # CNOT con qubit 0 (ψ) como control y qubit 1 (A) como objetivo
    # En base |ψ,A,B⟩: CNOT_{0→1} ⊗ I_B
    CNOT_01 = np.kron(CNOT, np.eye(2))  # actúa sobre qubits 0,1; I sobre qubit 2
    # H en qubit 0: H ⊗ I ⊗ I
    H_II = np.kron(np.kron(H, np.eye(2)), np.eye(2))
    
    # Aplicar circuito de Alice
    state = CNOT_01 @ initial
    state = H_II @ state
    
    # Proyectores de medida sobre qubits 0 y 1 de Alice
    results = {}
    for m1 in range(2):
        for m2 in range(2):
            # Proyector |m1,m2⟩⟨m1,m2| ⊗ I_B
            proj_a = np.zeros(4)  # proyector sobre qubits 0,1
            proj_a[m1*2 + m2] = 1.0
            proj_full = np.kron(np.outer(proj_a, proj_a), np.eye(2))
            
            projected = proj_full @ state
            prob = np.real(projected.conj() @ projected)
            
            if prob > 1e-10:
                # Estado de Bob (no normalizado)
                bob_unnorm = projected.reshape(4, 2)  # (qubit01, qubitB)
                # Solo tomamos la columna correspondiente a la medida
                bob_state = bob_unnorm[m1*2 + m2]
                bob_state = bob_state / np.linalg.norm(bob_state)
                
                # Corrección de Bob: X^m2 Z^m1
                correction = np.linalg.matrix_power(X, m2) @ np.linalg.matrix_power(Z, m1)
                bob_corrected = correction @ bob_state
                
                results[(m1, m2)] = {
                    'prob': prob,
                    'bob_raw': bob_state,
                    'bob_corrected': bob_corrected,
                    'equals_psi': np.allclose(bob_corrected, psi, atol=1e-10)
                }
    
    return results


# Teleportar |+⟩ = (|0⟩+|1⟩)/√2
psi_to_teleport = plus
print(f"Teleportando |ψ⟩ = {dirac_notation(psi_to_teleport)}")
print()

results = quantum_teleportation(psi_to_teleport)
print(f"{'Medida (m1,m2)':<18} {'Prob':<8} {'Estado Bob (antes corr.)':<30} {'Correcto'}")
print('-' * 70)
for (m1, m2), info in results.items():
    raw = dirac_notation(info['bob_raw'])
    print(f"  ({m1},{m2})           {info['prob']:<8.4f} {raw:<30} {info['equals_psi']}")

print()
print("Conclusión: en todos los casos, tras la corrección clásica X^m2 Z^m1,")
print(f"Bob recupera exactamente |ψ⟩ = {dirac_notation(psi_to_teleport)}")
print()
print("Nota: se han usado 2 bits clásicos + 1 par de Bell (1 ebit).")
print("El estado original de Alice se destruye (no-cloning theorem).")

### Ejercicio 5.1: Teleportar un estado arbitrario
Comprueba que el protocolo funciona para $|\psi\rangle = \frac{\sqrt{3}}{2}|0\rangle + \frac{i}{2}|1\rangle$.

In [ ]:
# Tu solución:
# psi_arb = qubit(math.sqrt(3)/2, 1j/2)
# print(f"Estado a teleportar: {dirac_notation(psi_arb)}")
# print(f"Norma: {np.linalg.norm(psi_arb):.4f}")
# results_arb = quantum_teleportation(psi_arb)
# for (m1, m2), info in results_arb.items():
#     print(f"Medida ({m1},{m2}): Bob obtiene {dirac_notation(info['bob_corrected'])} — correcto: {info['equals_psi']}")

print("TODO: teleporta el estado con componente imaginaria y verifica el resultado")

# Nota: el protocolo funciona para cualquier estado complejo, incluyendo
# estados con fase relativa imaginaria. La corrección Z ajusta la fase globalmente.

## 6. Comparación: entropía cuántica vs clásica

In [ ]:
# Diferencias clave entre entropía de Shannon y Von Neumann

print("Comparación: Shannon vs Von Neumann")
print("=" * 55)
print()

print("1. Estado puro |ψ⟩ (conocimiento completo):")
print("   Shannon: necesita distribución de probabilidad")
print("           si P(0)=|α|², P(1)=|β|², H = -Σ p log p")
rho_plus = density_matrix(plus)
H_shannon_plus = -(0.5 * math.log2(0.5) + 0.5 * math.log2(0.5))  # = 1 bit
VN_pure = von_neumann_entropy(rho_plus)
print(f"           H(medida de |+⟩) = {H_shannon_plus:.3f} bits")
print(f"   Von Neumann: S(|+⟩⟨+|) = {VN_pure:.3f} bits")
print(f"   → El estado puro |+⟩ tiene S=0 aunque su medida tenga H=1")
print(f"   → Von Neumann mide incertidumbre sobre el estado, no la medida")
print()

print("2. Subadicitividad: S(A,B) ≤ S(A) + S(B) para sistemas clásicos")
print("   En cuántico: S(A,B) puede ser < S(A) cuando están entrelazados")
rho_bell_ab = np.outer(phi_plus, phi_plus.conj())
rho_bell_a = partial_trace(rho_bell_ab, trace_out='B')
S_ab = von_neumann_entropy(rho_bell_ab)  # sistema puro: S=0
S_a = von_neumann_entropy(rho_bell_a)    # subsistema: S=1
print(f"   Para |Φ+⟩: S(AB) = {S_ab:.3f}, S(A) = {S_a:.3f}")
print(f"   → S(AB) < S(A): ¡paradoja clásicamente imposible!")
print(f"   → La información cuántica compartida puede ser negativa")
print()

print("3. No-cloning: es imposible copiar un estado cuántico desconocido")
print("   Consecuencia: la teleportación destruye el original")
print("   En clásico: copiar bits es trivial")
print()

print("Resumen:")
comparacion = [
    ("Objeto básico",     "bit (0 o 1)",          "qubit |ψ⟩ = α|0⟩+β|1⟩"),
    ("Entropía",          "H = -Σ p log p",        "S = -tr(ρ log ρ)"),
    ("Copiar",            "trivial",               "imposible (no-cloning)"),
    ("S(AB) ≤ S(A)+S(B)", "siempre",              "solo si separable"),
    ("S(estado puro)",    "S=H_b(p) ≥ 0",         "S=0 siempre"),
]
print(f"{'Propiedad':<25} {'Clásico':<30} {'Cuántico'}")
print('-' * 80)
for prop, clasico, cuantico in comparacion:
    print(f"{prop:<25} {clasico:<30} {cuantico}")

In [ ]:
# === Tests automáticos ===
import numpy as np
import math

# Test 1: qubit — el estado |0⟩ tiene probabilidades (1, 0)
ket0 = qubit(1, 0)
probs = measure_prob(ket0)
assert abs(probs[0] - 1.0) < 1e-10, f'P(0) esperado 1.0, obtenido {probs[0]}'
assert abs(probs[1] - 0.0) < 1e-10, f'P(1) esperado 0.0, obtenido {probs[1]}'

# Test 2: estado |+⟩ = H|0⟩ tiene probabilidades (0.5, 0.5)
H_gate = np.array([[1, 1], [1, -1]], dtype=complex) / math.sqrt(2)
plus_state = H_gate @ ket0
probs_plus = measure_prob(plus_state)
assert abs(probs_plus[0] - 0.5) < 1e-10, f'P(0) para |+⟩: esperado 0.5, obtenido {probs_plus[0]}'

# Test 3: HZH = X (identidad cuántica fundamental)
X_gate = np.array([[0, 1], [1, 0]], dtype=complex)
Z_gate = np.array([[1, 0], [0, -1]], dtype=complex)
HZH = H_gate @ Z_gate @ H_gate
assert np.allclose(HZH, X_gate), 'HZH debe ser igual a X'

# Test 4: entropía de Von Neumann de un estado puro = 0
rho_pure = density_matrix(ket0)
S_pure = von_neumann_entropy(rho_pure)
assert abs(S_pure) < 1e-10, f'Entropía de estado puro esperada 0, obtenida {S_pure}'

# Test 5: entropía de Von Neumann del estado máximamente mezclado = 1 (qubit)
rho_mixed = np.eye(2, dtype=complex) / 2
S_mixed = von_neumann_entropy(rho_mixed)
assert abs(S_mixed - 1.0) < 1e-10, f'Entropía de estado máximamente mezclado esperada 1.0, obtenida {S_mixed}'

# Test 6: teleportación cuántica preserva el estado
# Teleportar |+⟩ debe recuperar |+⟩
psi_in = plus_state.copy()
psi_out = quantum_teleportation(psi_in)
fidelity = abs(np.dot(psi_in.conj(), psi_out))**2
assert fidelity > 0.99, f'Fidelidad de teleportación: esperada ≥0.99, obtenida {fidelity:.4f}'

print('✓ Todos los tests de informacion-cuantica pasaron correctamente.')